# FIFA World Cup 2026 — Objective 2
## Comparison of Regression Models, Conclusion and Recommendations
**Scope:** Only the comparison and concluding sections of the supplied HIT140 report.

This notebook includes the **reported results** and optionally recomputes metrics from actual prediction CSVs. Reported values are not presented as newly computed results. The report does not provide individual cross-validation fold scores, so the notebook does **not** fabricate them.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display, Markdown
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

OUTPUT = Path('comparison_outputs')
OUTPUT.mkdir(exist_ok=True)
print('Libraries imported successfully.')

### 1. Comparison metrics reported in the document
Numbers below are transcribed from the provided report text. Missing exact results are kept as `NaN`, not guessed. The reported cross-validation R² is rounded to three decimal places.

In [ ]:
reported = pd.DataFrame({
    'Model': ['Regression 2.1', 'Regression 2.2'],
    'Target': ['Goal difference', 'Goals scored per team'],
    'Observations': [104, 208],
    'Test_R2': [-0.086, 0.234],
    'CV_Mean_R2': [0.133, 0.133],
    'CV_MAE': [1.422, 0.96],
    'Test_MAE': [np.nan, 0.99],
    'Target_SD': [2.02, 1.39],
    'CV_Fold_R2_SD': [0.28, 0.21],
    'Worst_CV_Fold_R2': [-0.20, -0.13]
})
reported['CV_MAE_Divided_By_Target_SD'] = reported['CV_MAE'] / reported['Target_SD']
display(reported.round(3))
reported.to_csv(OUTPUT / 'reported_regression_comparison.csv', index=False)

### 2. Visual comparison — test R², cross-validation R², and relative error
R² is unitless, but the models have different targets; the plots should not be interpreted as a controlled comparison. Normalizing MAE by target standard deviation is a rough scale adjustment, **not** proof that model performance is identical.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(reported))
w = 0.34
ax.bar(x-w/2, reported['Test_R2'], w, label='Test R²')
ax.bar(x+w/2, reported['CV_Mean_R2'], w, label='Mean CV R²')
ax.axhline(0, linewidth=1)
ax.set_xticks(x, reported['Model'])
ax.set_ylabel('R²')
ax.set_title('Reported R²: Test Split vs Five-Fold Cross-Validation')
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT / 'figure_5_r2_comparison.png', dpi=200)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(reported['Model'], reported['CV_MAE_Divided_By_Target_SD'])
ax.set_ylabel('CV MAE / target standard deviation')
ax.set_title('Scale-Adjusted Cross-Validation Error')
for b, v in zip(bars, reported['CV_MAE_Divided_By_Target_SD']):
    ax.text(b.get_x()+b.get_width()/2, b.get_height(), f'{v:.2f}', ha='center', va='bottom')
ax.set_ylim(0, 0.85)
fig.tight_layout()
fig.savefig(OUTPUT / 'normalized_cv_mae.png', dpi=200)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(reported['Model'], reported['CV_Fold_R2_SD'])
ax.set_ylabel('Reported SD of fold R²')
ax.set_title('Cross-Validation Stability (Lower = Less Variation)')
fig.tight_layout()
fig.savefig(OUTPUT / 'cv_stability_comparison.png', dpi=200)
plt.show()

### 3. Optional: import actual prediction CSV files and calculate metrics
If you have the real test predictions for each model, put `regression_2_1_predictions.csv` and `regression_2_2_predictions.csv` alongside the notebook. Each should have columns `actual` and `predicted`. These files are **optional**. Do not substitute training predictions for held-out test predictions. The notebook displays a message if the files are missing.

In [ ]:
def evaluate_predictions(path, model_name):
    path = Path(path)
    if not path.exists():
        print(f'{model_name}: {path.name} not found; using reported values only.')
        return None
    df = pd.read_csv(path)
    required = {'actual', 'predicted'}
    if not required.issubset(df.columns):
        raise ValueError(f'{path.name} must contain columns: actual, predicted')
    if df[list(required)].isna().any().any():
        raise ValueError(f'{path.name} contains missing predictions or actual values')
    y = pd.to_numeric(df['actual'], errors='raise')
    yhat = pd.to_numeric(df['predicted'], errors='raise')
    if len(df) < 2:
        raise ValueError('At least two prediction rows are needed.')
    result = {'Model': model_name, 'Rows': len(df),
              'MAE': mean_absolute_error(y, yhat),
              'RMSE': np.sqrt(mean_squared_error(y, yhat)),
              'R2': r2_score(y, yhat)}
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(y, yhat, alpha=0.7)
    low = min(y.min(), yhat.min()); high = max(y.max(), yhat.max())
    ax.plot([low, high], [low, high], linestyle='--')
    ax.set(xlabel='Actual', ylabel='Predicted', title=f'{model_name}: held-out predictions')
    fig.tight_layout()
    fig.savefig(OUTPUT / f'{model_name.replace(" ", "_")}_actual_predicted.png', dpi=200)
    plt.show()
    return result

actual_results = []
for name, filename in [('Regression 2.1','regression_2_1_predictions.csv'),
                       ('Regression 2.2','regression_2_2_predictions.csv')]:
    result = evaluate_predictions(filename, name)
    if result is not None:
        actual_results.append(result)
if actual_results:
    actual_table = pd.DataFrame(actual_results)
    display(actual_table.round(3))
    actual_table.to_csv(OUTPUT / 'recomputed_test_metrics.csv', index=False)

### 4. Comparison — interpretation from the supplied report

In [ ]:
display(Markdown('**Regression 2.2 performed better on the reported test split:** its test R² was **0.234**, compared with **−0.086** for Regression 2.1. However, both models had the same rounded mean five-fold cross-validation R² (**0.133**), so the test-split advantage did not carry over to average cross-validation performance.\n\nThe reported cross-validation MAE was **1.422 goals of goal difference** for Regression 2.1 and **0.96 goals scored** for Regression 2.2. Since the targets differ, raw MAE values are not directly comparable. Dividing by their respective target standard deviations gives approximately **0.70** and **0.69**.\n\nRegression 2.2 showed less variation between folds (reported fold-R² SD **0.21** vs **0.28**) and a less negative worst fold (**−0.13** vs **−0.20**). Ranking points were important predictors in Regression 2.2, while differences in recent form and scoring were important in Regression 2.1. The models also differ in features and observation structure, so these findings do not establish that team-level modelling is inherently superior.'))

### 5. Conclusion and recommendations

In [ ]:
display(Markdown('**Conclusion.** The two models captured limited but useful pre-match information. Their mean cross-validation R² values of approximately **0.133** indicate that the models explain a modest share of variation in their respective outcomes under the reported validation procedure. Regression 2.2 is the preferred model for this project because it had a stronger reported test R², less fold-to-fold variation and more interpretable ranking-based predictors. Nevertheless, its predicted goals should be treated as estimates, not exact scores.\n\n**Recommendations.** (1) Include a pre-match ranking-points difference in a revised Regression 2.1, while maintaining exactly eight explanatory variables by replacing an existing feature; (2) use repeated shuffled validation and group observations by match for Regression 2.2 to avoid splitting the two sides of the same fixture; (3) compare linear regression with **Poisson regression** for non-negative goal counts; and (4) consider verified pre-match line-ups, injuries, squad values and historical expected goals as additional candidate features. All features must be available before kickoff.\n\n**Caution.** The 208 rows in Regression 2.2 correspond to 104 paired matches, not 208 independent fixtures. Its larger row count should not be treated as automatically doubling the amount of independent evidence.'))

### 6. Export a comparison summary

In [ ]:
summary = f"""FIFA World Cup 2026 — Objective 2: Regression Comparison

Regression 2.1: test R² = -0.086; mean CV R² = 0.133; CV MAE = 1.422.
Regression 2.2: test R² = 0.234; mean CV R² = 0.133; CV MAE = 0.96.
Normalized CV MAE: {reported.loc[0, 'CV_MAE_Divided_By_Target_SD']:.2f} vs {reported.loc[1, 'CV_MAE_Divided_By_Target_SD']:.2f}.
Fold R² standard deviation: 0.28 vs 0.21.

Conclusion: Regression 2.2 is preferred on stability and reported test performance,
although both models have the same rounded average CV R² and limited predictive power.

Recommendations: replace a feature in 2.1 with ranking-points difference; use
repeated and match-grouped validation; test Poisson regression for goal counts;
add richer, verified pre-match data.

Source: supplied HIT140 report text; figures above are reported, not recomputed.
"""
(OUTPUT / 'comparison_conclusion_recommendations.txt').write_text(summary, encoding='utf-8')
print('Files saved to:', OUTPUT.resolve())
print('\n'.join(x.name for x in OUTPUT.iterdir()))